In [4]:
import dimod
import numpy as np

# no neal!

weights = [23,31,29,44,53,38,63,85,89,82]
values  = [92,57,49,68,60,43,67,84,87,72]
W = 165

def add_to_Q(Q, i, j, val):
    if i > j:
        i, j = j, i
    Q[(i, j)] = Q.get((i, j), 0) + val


def build_qubo(lam):
    Q = {}

    slack_coeffs = [1,2,4,8,16,32,64,128]
    coeffs = weights + slack_coeffs

    n = len(weights)
    m = len(slack_coeffs)

    # value term
    for i in range(n):
        add_to_Q(Q, i, i, -values[i])

    # penalty
    for i in range(n+m):
        ai = coeffs[i]
        add_to_Q(Q, i, i, lam*(ai**2 - 2*W*ai))

    for i in range(n+m):
        for j in range(i+1, n+m):
            ai = coeffs[i]
            aj = coeffs[j]
            add_to_Q(Q, i, j, lam*2*ai*aj)

    return Q


lam = 10
Q = build_qubo(lam)

bqm = dimod.BQM.from_qubo(Q)

result = dimod.ExactSolver().sample(bqm)

best = result.first.sample

x = [int(best[i]) for i in range(10)]

weight = sum(int(weights[i]) * int(x[i]) for i in range(10))
value  = sum(int(values[i]) * int(x[i]) for i in range(10))

print("x bits:", x)
print("選到:", [i+1 for i, bit in enumerate(x) if bit == 1])
print("weight:", weight)
print("value:", value)

# slack 也一起看
slack_coeffs = [1,2,4,8,16,32,64,128]
y = [int(best[10+j]) for j in range(8)]
slack = sum(slack_coeffs[j] * y[j] for j in range(8))

print("slack bits:", y)
print("slack:", slack)
print("weight + slack:", weight + slack)
print("feasible:", weight <= 165)

for lam in [0.01, 0.1, 1, 5, 10]:
    Q = build_qubo(lam)
    bqm = dimod.BQM.from_qubo(Q)
    result = dimod.ExactSolver().sample(bqm)
    best = result.first.sample

    x = [int(best[i]) for i in range(10)]
    weight = sum(weights[i] * x[i] for i in range(10))
    value = sum(values[i] * x[i] for i in range(10))

    print("lambda =", lam)
    print("selected:", [i+1 for i, bit in enumerate(x) if bit == 1])
    print("weight:", weight)
    print("value:", value)
    print("feasible:", weight <= 165)
    print()

x bits: [1, 1, 1, 1, 0, 1, 0, 0, 0, 0]
選到: [1, 2, 3, 4, 6]
weight: 165
value: 309
slack bits: [0, 0, 0, 0, 0, 0, 0, 0]
slack: 0
weight + slack: 165
feasible: True
lambda = 0.01
selected: [1, 2, 3, 4, 5, 6]
weight: 218
value: 369
feasible: False

lambda = 0.1
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309
feasible: True

lambda = 1
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309
feasible: True

lambda = 5
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309
feasible: True

lambda = 10
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309
feasible: True



In [8]:
import dimod
import neal
import time

# -----------------------------
# Problem data
# -----------------------------
weights = [23,31,29,44,53,38,63,85,89,82]
values  = [92,57,49,68,60,43,67,84,87,72]
W = 165

n_items = len(weights)
slack_coeffs = [1,2,4,8,16,32,64,128]
n_slack = len(slack_coeffs)
n_vars = n_items + n_slack

# -----------------------------
# Build QUBO (re-declared)
# -----------------------------
def build_qubo(lam):
    Q = {}
    coeffs = weights + slack_coeffs

    def add_to_Q(i, j, val):
        if i > j:
            i, j = j, i
        Q[(i, j)] = Q.get((i, j), 0) + val

    # value term
    for i in range(n_items):
        add_to_Q(i, i, -values[i])

    # penalty term
    for i in range(n_vars):
        ai = coeffs[i]
        add_to_Q(i, i, lam * (ai**2 - 2 * W * ai))

    for i in range(n_vars):
        for j in range(i + 1, n_vars):
            ai = coeffs[i]
            aj = coeffs[j]
            add_to_Q(i, j, lam * 2 * ai * aj)

    return Q

# -----------------------------
# Decode sample (re-declared)
# -----------------------------
def decode_sample(sample):
    x = [int(sample[i]) for i in range(n_items)]
    y = [int(sample[n_items + j]) for j in range(n_slack)]

    total_weight = sum(weights[i] * x[i] for i in range(n_items))
    total_value = sum(values[i] * x[i] for i in range(n_items))
    slack = sum(slack_coeffs[j] * y[j] for j in range(n_slack))

    feasible = total_weight <= W

    return x, y, total_weight, total_value, slack, feasible

# -----------------------------
# Build BQM
# -----------------------------
lam = 10
Q = build_qubo(lam)
bqm = dimod.BQM.from_qubo(Q)

# -----------------------------
# Reference optimal (ExactSolver)
# -----------------------------
exact_result = dimod.ExactSolver().sample(bqm)
best_exact = exact_result.first.sample

optimal_x, _, optimal_weight, optimal_value, _, _ = decode_sample(best_exact)

print("Reference optimal solution:")
print("selected:", [i+1 for i, bit in enumerate(optimal_x) if bit == 1])
print("weight:", optimal_weight)
print("value:", optimal_value)
print()

# -----------------------------
# Part C: Simulated Annealing
# -----------------------------
sampler = neal.SimulatedAnnealingSampler()

num_reads_list = [10, 100, 1000, 10000]

print("========== Part C: Simulated Annealing ==========")

for num_reads in num_reads_list:
    start = time.time()

    sampleset = sampler.sample(
        bqm,
        num_reads=num_reads,
        seed=13202050
    )

    elapsed = time.time() - start

    success_count = 0
    best_value_found = -1
    best_weight_found = None
    best_x_found = None
    best_energy_found = None

    for sample, energy in sampleset.data(["sample", "energy"]):
        x, y, weight, value, slack, feasible = decode_sample(sample)

        # success = found optimal solution
        if x == optimal_x:
            success_count += 1

        # best feasible solution
        if feasible and value > best_value_found:
            best_value_found = value
            best_weight_found = weight
            best_x_found = x
            best_energy_found = energy

    success_probability = success_count / num_reads

    print()
    print("num_reads =", num_reads)
    print("success count =", success_count)
    print("success probability =", success_probability)
    print("best selected:", [i+1 for i, bit in enumerate(best_x_found) if bit == 1])
    print("best weight found:", best_weight_found)
    print("best value found:", best_value_found)
    print("best energy found:", best_energy_found)
    print("time:", elapsed)

Reference optimal solution:
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309

========== Part C: Simulated Annealing ==========

num_reads = 10
success count = 0
success probability = 0.0
best selected: [1, 4, 8]
best weight found: 152
best value found: 244
best energy found: -272494.0
time: 0.014595985412597656

num_reads = 100
success count = 0
success probability = 0.0
best selected: [1, 3, 4, 7]
best weight found: 159
best value found: 276
best energy found: -272526.0
time: 0.017657756805419922

num_reads = 1000
success count = 4
success probability = 0.004
best selected: [1, 2, 3, 4, 6]
best weight found: 165
best value found: 309
best energy found: -272559.0
time: 0.17120623588562012

num_reads = 10000
success count = 43
success probability = 0.0043
best selected: [1, 2, 3, 4, 6]
best weight found: 165
best value found: 309
best energy found: -272559.0
time: 1.7385437488555908


In [7]:
import dimod
import numpy as np
import time   # ← 加這行

weights = [23,31,29,44,53,38,63,85,89,82]
values  = [92,57,49,68,60,43,67,84,87,72]
W = 165

def add_to_Q(Q, i, j, val):
    if i > j:
        i, j = j, i
    Q[(i, j)] = Q.get((i, j), 0) + val


def build_qubo(lam):
    Q = {}

    slack_coeffs = [1,2,4,8,16,32,64,128]
    coeffs = weights + slack_coeffs

    n = len(weights)
    m = len(slack_coeffs)

    # value term
    for i in range(n):
        add_to_Q(Q, i, i, -values[i])

    # penalty
    for i in range(n+m):
        ai = coeffs[i]
        add_to_Q(Q, i, i, lam*(ai**2 - 2*W*ai))

    for i in range(n+m):
        for j in range(i+1, n+m):
            ai = coeffs[i]
            aj = coeffs[j]
            add_to_Q(Q, i, j, lam*2*ai*aj)

    return Q


# =============================
# ExactSolver（主要結果）
# =============================
lam = 10
Q = build_qubo(lam)
bqm = dimod.BQM.from_qubo(Q)

start = time.time()   # ← 計時開始
result = dimod.ExactSolver().sample(bqm)
elapsed = time.time() - start   # ← 計時結束

best = result.first.sample

x = [int(best[i]) for i in range(10)]

weight = sum(int(weights[i]) * int(x[i]) for i in range(10))
value  = sum(int(values[i]) * int(x[i]) for i in range(10))

print("=== ExactSolver result ===")
print("x bits:", x)
print("選到:", [i+1 for i, bit in enumerate(x) if bit == 1])
print("weight:", weight)
print("value:", value)
print("time:", elapsed)   # ← 新增
print()

# slack
slack_coeffs = [1,2,4,8,16,32,64,128]
y = [int(best[10+j]) for j in range(8)]
slack = sum(slack_coeffs[j] * y[j] for j in range(8))

print("slack bits:", y)
print("slack:", slack)
print("weight + slack:", weight + slack)
print("feasible:", weight <= 165)
print()


# =============================
# Lambda 測試（含時間）
# =============================
print("=== Lambda comparison ===")

for lam in [0.01, 0.1, 1, 5, 10]:
    Q = build_qubo(lam)
    bqm = dimod.BQM.from_qubo(Q)

    start = time.time()   # ← 計時開始
    result = dimod.ExactSolver().sample(bqm)
    elapsed = time.time() - start   # ← 計時結束

    best = result.first.sample

    x = [int(best[i]) for i in range(10)]
    weight = sum(weights[i] * x[i] for i in range(10))
    value = sum(values[i] * x[i] for i in range(10))

    print("lambda =", lam)
    print("selected:", [i+1 for i, bit in enumerate(x) if bit == 1])
    print("weight:", weight)
    print("value:", value)
    print("feasible:", weight <= 165)
    print("time:", elapsed)   # ← 新增
    print()

=== ExactSolver result ===
x bits: [1, 1, 1, 1, 0, 1, 0, 0, 0, 0]
選到: [1, 2, 3, 4, 6]
weight: 165
value: 309
time: 0.18966889381408691

slack bits: [0, 0, 0, 0, 0, 0, 0, 0]
slack: 0
weight + slack: 165
feasible: True

=== Lambda comparison ===
lambda = 0.01
selected: [1, 2, 3, 4, 5, 6]
weight: 218
value: 369
feasible: False
time: 0.17502379417419434

lambda = 0.1
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309
feasible: True
time: 0.17798089981079102

lambda = 1
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309
feasible: True
time: 0.1969590187072754

lambda = 5
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309
feasible: True
time: 0.1761331558227539

lambda = 10
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309
feasible: True
time: 0.17645907402038574



In [ ]:
import dimod
import neal
import time

# -----------------------------
# Problem data
# -----------------------------
weights = [23,31,29,44,53,38,63,85,89,82]
values  = [92,57,49,68,60,43,67,84,87,72]
W = 165

n_items = len(weights)
slack_coeffs = [1,2,4,8,16,32,64,128]
n_slack = len(slack_coeffs)
n_vars = n_items + n_slack

# -----------------------------
# Build QUBO (re-declared)
# -----------------------------
def build_qubo(lam):
    Q = {}
    coeffs = weights + slack_coeffs

    def add_to_Q(i, j, val):
        if i > j:
            i, j = j, i
        Q[(i, j)] = Q.get((i, j), 0) + val

    # value term
    for i in range(n_items):
        add_to_Q(i, i, -values[i])

    # penalty term
    for i in range(n_vars):
        ai = coeffs[i]
        add_to_Q(i, i, lam * (ai**2 - 2 * W * ai))

    for i in range(n_vars):
        for j in range(i + 1, n_vars):
            ai = coeffs[i]
            aj = coeffs[j]
            add_to_Q(i, j, lam * 2 * ai * aj)

    return Q

# -----------------------------
# Decode sample (re-declared)
# -----------------------------
def decode_sample(sample):
    x = [int(sample[i]) for i in range(n_items)]
    y = [int(sample[n_items + j]) for j in range(n_slack)]

    total_weight = sum(weights[i] * x[i] for i in range(n_items))
    total_value = sum(values[i] * x[i] for i in range(n_items))
    slack = sum(slack_coeffs[j] * y[j] for j in range(n_slack))

    feasible = total_weight <= W

    return x, y, total_weight, total_value, slack, feasible

# -----------------------------
# Build BQM
# -----------------------------
lam = 10
Q = build_qubo(lam)
bqm = dimod.BQM.from_qubo(Q)

# -----------------------------
# Reference optimal (ExactSolver)
# -----------------------------
exact_result = dimod.ExactSolver().sample(bqm)
best_exact = exact_result.first.sample

optimal_x, _, optimal_weight, optimal_value, _, _ = decode_sample(best_exact)

print("Reference optimal solution:")
print("selected:", [i+1 for i, bit in enumerate(optimal_x) if bit == 1])
print("weight:", optimal_weight)
print("value:", optimal_value)
print()

# -----------------------------
# Part C: Simulated Annealing
# -----------------------------
sampler = neal.SimulatedAnnealingSampler()

num_reads_list = [10, 100, 1000, 10000]

print("========== Part C: Simulated Annealing ==========")

for num_reads in num_reads_list:
    start = time.time()

    sampleset = sampler.sample(
        bqm,
        num_reads=num_reads,
        seed=13202050
    )

    elapsed = time.time() - start

    success_count = 0
    best_value_found = -1
    best_weight_found = None
    best_x_found = None
    best_energy_found = None

    for sample, energy in sampleset.data(["sample", "energy"]):
        x, y, weight, value, slack, feasible = decode_sample(sample)

        # success = found optimal solution
        if x == optimal_x:
            success_count += 1

        # best feasible solution
        if feasible and value > best_value_found:
            best_value_found = value
            best_weight_found = weight
            best_x_found = x
            best_energy_found = energy

    success_probability = success_count / num_reads

    print()
    print("num_reads =", num_reads)
    print("success count =", success_count)
    print("success probability =", success_probability)
    print("best selected:", [i+1 for i, bit in enumerate(best_x_found) if bit == 1])
    print("best weight found:", best_weight_found)
    print("best value found:", best_value_found)
    print("best energy found:", best_energy_found)
    print("time:", elapsed)

Reference optimal solution:
selected: [1, 2, 3, 4, 6]
weight: 165
value: 309

========== Part C: Simulated Annealing ==========

num_reads = 10
success count = 0
success probability = 0.0
best selected: [1, 4, 8]
best weight found: 152
best value found: 244
best energy found: -272494.0
time: 0.014595985412597656

num_reads = 100
success count = 0
success probability = 0.0
best selected: [1, 3, 4, 7]
best weight found: 159
best value found: 276
best energy found: -272526.0
time: 0.017657756805419922

num_reads = 1000
success count = 4
success probability = 0.004
best selected: [1, 2, 3, 4, 6]
best weight found: 165
best value found: 309
best energy found: -272559.0
time: 0.17120623588562012

num_reads = 10000
success count = 43
success probability = 0.0043
best selected: [1, 2, 3, 4, 6]
best weight found: 165
best value found: 309
best energy found: -272559.0
time: 1.7385437488555908
